# מסע אל המשנה — שני אודישנים ב-VoxCPM2

מחברת גיבוי בלבד; המשתמש אינו נדרש להפעיל אותה כדי שהעוזר ינסה בסביבת הענן שלו. הקוד זהה לקובץ Python שבמאגר, עם חבילה2.0.3 ומשקולות מוצמדות.

Voice Design יוצר קול סינתטי מתיאור; הגרסה השנייה נשענת רק על הקול הסינתטי הראשון שלנו. אין SASPEECH או הקלטת אדם לשיבוט. רישיון המודל לפי OpenBMB הוא Apache2.0; אין אישור ליהוק/תוכן/פרסום. גיל, עברית טבעית, יציבות ורגש מחייבים בדיקת האזנה.

Colab דורש GPU חינמי זמין. בדיקת CUDA אינה אימות שכבת חיוב: אין לבחור מסלול בתשלום, לרכוש מכסה או לשדרג. בלי GPU זמין עוצרים. אין הבטחה למכסה חינמית. אין ניסיונות סינתזה אוטומטיים חוזרים או חימום שמייצר משפט נוסף.


In [ ]:
import sys, subprocess, torch
if not torch.cuda.is_available():
    raise RuntimeError("No existing free GPU; stop without purchasing or upgrading.")
gpu = torch.cuda.get_device_properties(0)
print("GPU:", gpu.name, "VRAM GiB:", round(gpu.total_memory / 2**30, 1))
if gpu.total_memory < 8 * 2**30:
    raise RuntimeError("Less than8GB VRAM; stop, do not upgrade.")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "voxcpm==2.0.3"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "soundfile", "safetensors", "huggingface-hub", "transformers>=4.36.2,<6", "einops", "pydantic>=2", "librosa", "tqdm", "addict", "inflect", "sortedcontainers"])


In [ ]:
from pathlib import Path
import subprocess, sys
# Embedded from the one repository experiment, not another synthesis implementation.
SCRIPT = '"""Bounded VoxCPM2 research; no production defaults, paid API or human voice input.\n\nInstall runtime: voxcpm==2.0.3 (CPU/CUDA torch matching the environment).\n--plan uses only Python\'s standard library and never downloads or synthesizes.\n"""\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport inspect\nimport json\nimport os\nimport random\nimport signal\nimport time\n\nMODEL_ID = \'openbmb/VoxCPM2\'\nMODEL_REVISION = \'32279effe8c19989596f05d353d1447f51d9e915\'\nPACKAGE_VERSION = \'2.0.3\'\nEXPECTED_HASHES = {\'audiovae.pth\': \'94b5d51e107e0507d4acc976cfdadb64edd6fd06d1f751dadbf2fd1594274bf1\', \'model.safetensors\': \'f7f964cfa9da23653baec6e6f7750719977ad944ed9f95fe52fe3a620506891d\'}\nSEED = 731\nHEBREW_TEXT = (\n    "מאחורי מילות המשנה עומד עולם רחוק. "\n    "חכמים למדו, שאלו, והעבירו את המסורת מדור לדור. "\n    "כעת נפתח במסכת ברכות, ונשאל שאלה פשוטה: "\n    "מתי קוראים את קריאת שמע? "\n    "רבן גמליאל אומר: עד שיעלה עמוד השחר. "\n    "אבל מה משמעות הדברים? כדי להבין, נחזור אל ראשית הסיפור."\n)\nVARIANTS = (\n    {\'id\': \'warm\', \'description\': (\n        \'A mature male native Israeli Hebrew speaker around 65, a kind wise scholar, \'\n        \'warm rich natural baritone, healthy steady voice, affectionate gentle smile, \'\n        \'patient confident storytelling, clear effortless Hebrew and nuanced emotion.\')},\n    {\'id\': \'dramatic\', \'description\': (\n        \'The same mature male native Israeli Hebrew speaker around 65, warm and kind, \'\n        \'slightly richer natural chest resonance, quiet cinematic gravitas, thoughtful \'\n        \'pauses, restrained emotional storytelling and clear effortless Hebrew.\')},\n)\n\n\ndef generation_options(generate_function, seed):\n    """2.0.3 has no seed kwarg; newer upstream APIs do. Never request retries."""\n    options = dict(cfg_value=2.0, inference_timesteps=10, max_len=1024,\n                   normalize=False, denoise=False, retry_badcase=False,\n                   retry_badcase_max_times=1)\n    if \'seed\' in inspect.signature(generate_function).parameters:\n        options[\'seed\'] = seed\n    return options\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--plan\', action=\'store_true\')\n    parser.add_argument(\'--device\', choices=[\'cpu\', \'cuda\', \'auto\'], default=\'auto\')\n    parser.add_argument(\'--model-dir\', type=Path)\n    parser.add_argument(\'--output-dir\', type=Path, default=Path(\'build/voxcpm2-research\'))\n    parser.add_argument(\'--variant\', choices=[\'warm\', \'dramatic\', \'both\'], default=\'both\')\n    args = parser.parse_args(argv)\n    plan = dict(model=MODEL_ID, revision=MODEL_REVISION, package=PACKAGE_VERSION,\n                text=HEBREW_TEXT, variants=list(VARIANTS), seed=SEED,\n                generation_started=False, narrator_approved=False,\n                publication_approved=False, human_voice_reference=None)\n    if args.plan:\n        print(json.dumps(plan, ensure_ascii=False, indent=2))\n        return 0\n    # Heavy dependencies remain inside explicit execution, never import/--plan.\n    import importlib.metadata\n    import numpy as np\n    import soundfile as sf\n    import torch\n    from voxcpm import VoxCPM\n    if importlib.metadata.version(\'voxcpm\') != PACKAGE_VERSION:\n        raise RuntimeError(f\'This experiment was checked with voxcpm=={PACKAGE_VERSION}\')\n    torch.set_num_threads(4)\n    device = (\'cuda\' if torch.cuda.is_available() else \'cpu\') if args.device == \'auto\' else args.device\n    if device == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA is unavailable; no paid resource will be provisioned\')\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    execution = dict(plan, device=device, stage=\'model_files\', status=\'running\',\n                     backend_dtype=\'checkpoint default; no quantization override\')\n    report_path = args.output_dir / \'execution.json\'\n    def persist():\n        report_path.write_text(json.dumps(execution, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    def interrupted(_signum, _frame):\n        raise TimeoutError(\'External runtime bound reached; no automatic synthesis retry\')\n    signal.signal(signal.SIGTERM, interrupted)\n    persist()\n    try:\n        if args.model_dir:\n            model_path = args.model_dir\n        else:\n            from huggingface_hub import snapshot_download\n            model_path = Path(snapshot_download(MODEL_ID, revision=MODEL_REVISION,\n                                               allow_patterns=[\'*.json\', \'*.py\', \'*.safetensors\', \'audiovae.pth\']))\n        for name, expected in EXPECTED_HASHES.items():\n            with (model_path / name).open(\'rb\') as handle:\n                actual = hashlib.file_digest(handle, \'sha256\').hexdigest()\n            if actual != expected:\n                raise ValueError(f\'Pinned checkpoint hash mismatch: {name}\')\n        execution[\'stage\'] = \'load_model\'\n        persist()\n        print(\'Loading pinned VoxCPM2 on\', device, \'without compile/warm-up\', flush=True)\n        model = VoxCPM.from_pretrained(str(model_path), device=device,\n                                       load_denoiser=False, optimize=False)\n        sr = int(model.tts_model.sample_rate)\n        if sr != 48000:\n            raise ValueError(f\'Unexpected VoxCPM2 output rate: {sr}; refusing incorrect playback metadata\')\n        execution[\'backend_dtype\'] = str(model.tts_model.config.dtype)\n        options = generation_options(model._generate, SEED)\n        execution[\'options\'] = options\n        execution[\'seed_method\'] = \'native seed argument\' if \'seed\' in options else \'random/numpy/torch global seed before each call\'\n        execution[\'outputs\'] = []\n        for variant in VARIANTS:\n            if args.variant != \'both\' and args.variant != variant[\'id\']:\n                continue\n            output = args.output_dir / (variant[\'id\'] + \'.wav\')\n            sidecar = output.with_suffix(\'.json\')\n            request = dict(model=MODEL_ID, revision=MODEL_REVISION, package=PACKAGE_VERSION,\n                           text=HEBREW_TEXT, voice_description=variant[\'description\'], seed=SEED,\n                           options=options, device=device)\n            # New design first; second conditions only on our OWN synthetic first voice.\n            reference = args.output_dir / \'warm.wav\' if variant[\'id\'] == \'dramatic\' else None\n            if reference and not reference.exists():\n                raise FileNotFoundError(\'Create warm first; no human/dataset fallback reference is allowed\')\n            request[\'synthetic_reference_sha256\'] = hashlib.sha256(reference.read_bytes()).hexdigest() if reference else None\n            if output.exists() or sidecar.exists():\n                if not (output.exists() and sidecar.exists()):\n                    raise ValueError(\'Incomplete prior result; preserve it for investigation, do not overwrite\')\n                prior = json.loads(sidecar.read_text())\n                if prior[\'request\'] != request or hashlib.sha256(output.read_bytes()).hexdigest() != prior[\'sha256\']:\n                    raise ValueError(\'Prior result differs or is corrupt; no automatic rerun\')\n                execution[\'outputs\'].append(prior)\n                continue\n            execution[\'stage\'] = variant[\'id\']\n            execution[\'generation_started\'] = True\n            persist()\n            print(\'Generating\', variant[\'id\'], \'once; retries disabled\', flush=True)\n            random.seed(SEED)\n            np.random.seed(SEED)\n            torch.manual_seed(SEED)\n            if torch.cuda.is_available():\n                torch.cuda.manual_seed_all(SEED)\n            started = time.monotonic()\n            wave = np.asarray(model.generate(text=f"({variant[\'description\']}){HEBREW_TEXT}",\n                                             reference_wav_path=str(reference) if reference else None,\n                                             **options), dtype=np.float32)\n            if wave.ndim != 1 or wave.size < sr or not np.isfinite(wave).all():\n                raise ValueError(\'Invalid, empty, too short or non-finite waveform\')\n            peak = float(np.max(np.abs(wave)))\n            if peak <= 1e-5:\n                raise ValueError(\'No audible signal generated\')\n            sf.write(output.with_name(variant[\'id\'] + \'-raw-float.wav\'), wave, sr, subtype=\'FLOAT\')\n            gain = min(1.0, .95 / peak)\n            sf.write(output, wave * gain, sr, subtype=\'PCM_16\')\n            item = dict(request=request, sample_rate=sr, samples=len(wave), seconds=len(wave)/sr,\n                        raw_peak=peak, static_gain=float(gain), elapsed_seconds=time.monotonic()-started,\n                        sha256=hashlib.sha256(output.read_bytes()).hexdigest(),\n                        evaluated_by_human=False, narrator_approved=False, publication_approved=False,\n                        processing=\'global gain only; raw float preserved, no Pitch/EQ/time stretch\')\n            sidecar.write_text(json.dumps(item, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n            execution[\'outputs\'].append(item)\n            persist()\n            print(\'Saved\', output, item[\'seconds\'], \'seconds\', flush=True)\n        execution.update(status=\'completed\', stage=\'saved\')\n        persist()\n        return 0\n    except Exception as exc:\n        execution.update(status=\'failed\', error_type=type(exc).__name__, error=str(exc))\n        persist()\n        raise\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n'
script_path = Path("/content/voxcpm2_mature_hebrew_research.py")
script_path.write_text(SCRIPT, encoding="utf-8")
subprocess.run(["timeout", "-k", "15", "900", sys.executable, str(script_path), "--device", "cuda", "--output-dir", "/content/voxcpm2-audition"], check=True)


In [ ]:
from pathlib import Path
import json, zipfile
from IPython.display import display, Audio
from google.colab import files
root = Path("/content/voxcpm2-audition")
report = json.loads((root / "execution.json").read_text())
if report["status"] != "completed" or len(report["outputs"]) != 2:
    raise RuntimeError("Two completed outputs not verified; no stale archive offered.")
for name in ["warm", "dramatic"]:
    print(name, "— unapproved research voice")
    display(Audio(filename=str(root / (name + ".wav"))))
archive = root / "voxcpm2-private-audition.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for file in root.iterdir():
        if file.suffix in [".wav", ".json"]:
            z.write(file, file.name)
files.download(str(archive))
